In [ ]:
#| default_exp _tokenizer

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import torch
from dreamer4._core import *
import matplotlib.pyplot as plt

In [ ]:
#| export
from dreamer4 import VideoTokenizer

@torch.no_grad()
def show_reconstructions(
    model: VideoTokenizer,
    video: torch.Tensor,
    title_prefix: str = "",
    max_time: int = 4,
):
    """
    Show original vs reconstruction for a single video batch element.
    """
    device = next(model.parameters()).device
    model.eval()

    video = batchify_video(video)
    video = video.to(device)

    recon = model.decode(
        model.tokenize(video),
        height=video.shape[-2],
        width=video.shape[-1],
    )

    video = video[0].cpu()   # (c, t, h, w)
    recon = recon[0].cpu()

    c, t, h, w = video.shape
    t_show = min(t, max_time)

    fig, axes = plt.subplots(2, t_show, figsize=(3 * t_show, 6))

    for i in range(t_show):
        axes[0, i].imshow(video[:, i].permute(1, 2, 0).clamp(0, 1))
        axes[0, i].set_title(f"{title_prefix}orig t={i}")
        axes[0, i].axis("off")

        axes[1, i].imshow(recon[:, i].permute(1, 2, 0).clamp(0, 1))
        axes[1, i].set_title(f"{title_prefix}recon t={i}")
        axes[1, i].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
from dreamer4._core import build_tiny_pinpad_dataset
device = 'cuda'
dataset, _ = build_tiny_pinpad_dataset(device=device, episode_length=30)

In [ ]:
tiny_loader = make_tiny_dataloader(dataset, n_items=1024, batch_size=16)
lpips_loss = 0.0
tokenizer = VideoTokenizer(
    dim=64,#16,
    # encoder_depth = 1, # I think the paper has a different depths and does the time blocks every n (8?) layers
    # decoder_depth = 1,
    # time_block_every = 1,
    dim_latent = 64, #16,
    patch_size = 16, #32
    attn_dim_head = 32, #16,
    num_latent_tokens = 4,
    lpips_loss_weight=lpips_loss,
).to(device)


In [ ]:

for _ in range(10):
    cfg = TinyOverfitConfig(steps=2000 if (lpips_loss <= 0) else 10000, mask_patches=True)
    losses = train_tiny_overfit(tokenizer, tiny_loader, cfg)
    plot_losses(losses)

    batch = next(iter(tiny_loader))
    video = batch[0] if isinstance(batch, (tuple, list)) else batch
    show_reconstructions(tokenizer, video, title_prefix=f"tiny-{_}-")

In [ ]:
#| export

import math
import torch.nn.functional as F


@torch.no_grad()
def debug_latents_and_errors(
    model: VideoTokenizer,
    video: torch.Tensor,
    patch_size: int | None = None,
    frame_index: int = 0,
):
    """
    - Prints latent norm/variance stats.
    - Shows pixel-wise error and patch-wise error heatmaps for one frame.
    """

    model.eval()
    video = batchify_video(video).to(model.device)      # (b,c,t,h,w)
    b, c, t, h, w = video.shape
    patch_size = patch_size or model.patch_size

    # --- latents ---
    latents = model.tokenize(video)                     # (b,t,n,d_latent)
    b, t, n, d_latent = latents.shape

    lat_mean = latents.mean(dim=(0,1,2))
    lat_std  = latents.std(dim=(0,1,2))
    lat_norm = latents.norm(dim=-1)                    # (b,t,n)

    print("latents shape:", latents.shape)
    print("latent dim:", d_latent, "num_latent_tokens:", n)
    print("latent per-dim mean (first 5):", lat_mean[:5].cpu().numpy())
    print("latent per-dim std  (first 5):", lat_std[:5].cpu().numpy())
    print("latent L2 norm stats: mean={:.3f}, std={:.3f}, min={:.3f}, max={:.3f}".format(
        lat_norm.mean().item(), lat_norm.std().item(),
        lat_norm.min().item(), lat_norm.max().item()
    ))

    # histogram of latent norms
    plt.figure(figsize=(4,3))
    plt.hist(lat_norm.flatten().cpu().numpy(), bins=40)
    plt.title("latent L2 norms")
    plt.xlabel("norm"); plt.ylabel("count")
    plt.tight_layout()
    plt.show()

    # --- reconstruction + error ---
    recon = model.decode(latents, height=h, width=w)
    recon = recon.clamp(0, 1)

    mse = F.mse_loss(video, recon).item()
    print("global MSE:", mse)

    vid0 = video[0, :, frame_index].cpu()              # (c,h,w)
    rec0 = recon[0, :, frame_index].cpu()

    err = (vid0 - rec0).pow(2).mean(dim=0)             # (h,w) per-pixel MSE

    # patch-wise error (mean over patch)
    ph, pw = h // patch_size, w // patch_size
    err_patch = err.view(ph, patch_size, pw, patch_size).mean(dim=(1,3))  # (ph,pw)

    # --- plots ---
    fig, axes = plt.subplots(1, 3, figsize=(12,4))

    axes[0].imshow(vid0.permute(1,2,0))
    axes[0].set_title("orig frame t={}".format(frame_index))
    axes[0].axis("off")

    axes[1].imshow(rec0.permute(1,2,0))
    axes[1].set_title("recon frame t={}".format(frame_index))
    axes[1].axis("off")

    im = axes[2].imshow(err, cmap="viridis")
    axes[2].set_title("per-pixel error")
    axes[2].axis("off")
    fig.colorbar(im, ax=axes[2])
    plt.tight_layout()
    plt.show()

    # patch error heatmap
    plt.figure(figsize=(4,4))
    plt.imshow(err_patch, cmap="magma")
    plt.title("per-patch error")
    plt.colorbar()
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    return latents.cpu(), vid0, rec0, err, err_patch

batch = next(iter(tiny_loader))
video = batch[0] if isinstance(batch, (tuple, list)) else batch

latents, vid0, rec0, err, err_patch = debug_latents_and_errors(
    tokenizer, video, frame_index=0
)


In [ ]:
@torch.no_grad()
def latent_probes(
    model: VideoTokenizer,
    video_batch: torch.Tensor,
    max_time: int = 4,
):
    device = model.device
    model.eval()

    video_batch = batchify_video(video_batch).to(device)   # (b,c,t,h,w)
    b, c, t, h, w = video_batch.shape

    latents = model.tokenize(video_batch)                  # (b,t,n,d)
    def decode(z): return model.decode(z, height=h, width=w)

    recon = decode(latents)
    zero_recon = decode(torch.zeros_like(latents))
    shuffle_recon = decode(latents[torch.randperm(b, device=device)])

    vid0 = video_batch[0].cpu()
    rec0 = recon[0].cpu()
    rec_zero0 = zero_recon[0].cpu()
    rec_shuffle0 = shuffle_recon[0].cpu()

    t_show = min(t, max_time)
    fig, axes = plt.subplots(4, t_show, figsize=(3*t_show, 8))
    rows = [
        (vid0, "orig"),
        (rec0, "recon"),
        (rec_zero0, "zero_latents"),
        (rec_shuffle0, "shuffled_latents"),
    ]
    for col in range(t_show):
        for row_idx, (tensor, title) in enumerate(rows):
            axes[row_idx, col].imshow(tensor[:, col].permute(1,2,0).clamp(0,1))
            axes[row_idx, col].set_title(f"{title} t={col}")
            axes[row_idx, col].axis("off")
    plt.tight_layout()
    plt.show()

batch = next(iter(tiny_loader))
video = batch[0] if isinstance(batch, (tuple, list)) else batch
latent_probes(tokenizer, video)



In [ ]:
@torch.no_grad()
def latent_effect_mse(model, video):
    model.eval()
    video = batchify_video(video).to(model.device)
    b, c, t, h, w = video.shape

    latents = model.tokenize(video)
    def decode(z): return model.decode(z, height=h, width=w)

    recon = decode(latents)
    zero_recon = decode(torch.zeros_like(latents))
    shuffle_recon = decode(latents[torch.randperm(b, device=model.device)])

    mse_true = F.mse_loss(video, recon).item()
    mse_zero = F.mse_loss(video, zero_recon).item()
    mse_shuffle = F.mse_loss(video, shuffle_recon).item()

    print("MSE(video, recon)        =", mse_true)
    print("MSE(video, zero_latents) =", mse_zero)
    print("MSE(video, shuffled)     =", mse_shuffle)

batch = next(iter(tiny_loader))
video = batch[0] if isinstance(batch, (tuple, list)) else batch
latent_effect_mse(tokenizer, video)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()